関数読み込み

In [12]:
%load_ext autoreload
%autoreload 2

import sys
import os

# 現在のフォルダの「1つ上の階層（../）」をシステムのパスに追加する
sys.path.append(os.path.abspath(".."))

# これでプロジェクト全体のフォルダが見えるようになるので、通常通りインポート可能
from src.data_prep import give_pair_id, give_mode_id, filter_valid_pairs, filter_lane,filter_mode,assign_and_count_pair_type



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### データ読み込みと結合

In [7]:
#csvファイルではなくtxtファイルを読み込む
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


path_1 = Path('../data/raw/trajectories-0400-0415.txt')
path_2 = Path('../data/raw/trajectories-0500-0515.txt')
path_3 = Path('../data/raw/trajectories-0515-0530.txt')

columns = [
    'Vehicle_ID', 'Frame_ID', 'Total_Frames', 'Global_Time', 'Local_X', 'Local_Y',
    'Global_X', 'Global_Y', 'v_Length', 'v_Width', 'v_Class', 'v_Vel', 'v_Acc',
    'Lane_ID', 'Preceding', 'Following', 'Space_Headway', 'Time_Headway'
]

df_1 = pd.read_csv(path_1, sep=r'\s+', header=None, names=columns)
df_2 = pd.read_csv(path_2, sep=r'\s+', header=None, names=columns)
df_3 = pd.read_csv(path_3, sep=r'\s+', header=None, names=columns)

# 時間帯の識別列を追加 (id の重複があるため)
df_1['time_period'] = 1  # 4:00-4:15
df_2['time_period'] = 2  # 5:00-5:15
df_3['time_period'] = 3  # 5:15-5:30

# 結合
df = pd.concat([df_1, df_2, df_3], ignore_index=True)

# データの確認
print(df.shape)
df.head()

(4566387, 19)


,Vehicle_ID,Frame_ID,Total_Frames,Global_Time,Local_X,Local_Y,Global_X,Global_Y,v_Length,v_Width,v_Class,v_Vel,v_Acc,Lane_ID,Preceding,Following,Space_Headway,Time_Headway,time_period
0,1,12,884,1113433136100,16.884,48.213,6042842.116,2133117.662,14.3,6.4,2,12.5,0.0,2,0,0,0.0,0.0,1
1,1,13,884,1113433136200,16.938,49.463,6042842.012,2133118.909,14.3,6.4,2,12.5,0.0,2,0,0,0.0,0.0,1
2,1,14,884,1113433136300,16.991,50.712,6042841.908,2133120.155,14.3,6.4,2,12.5,0.0,2,0,0,0.0,0.0,1
3,1,15,884,1113433136400,17.045,51.963,6042841.805,2133121.402,14.3,6.4,2,12.5,0.0,2,0,0,0.0,0.0,1
4,1,16,884,1113433136500,17.098,53.213,6042841.701,2133122.649,14.3,6.4,2,12.5,0.0,2,0,0,0.0,0.0,1


#### ペア・モードID付与、df_validの作成

In [ ]:
# テストケース
test_data = {
    'Vehicle_ID': [1, 1, 1, 1, 1, 1,   # 車両1: Car
                   2, 2, 2, 2, 2,        # 車両2: Truck
                   3, 3, 3, 3, 3],       # 車両3: Car
    'Global_Time': [100, 200, 300, 400, 500, 600,
                    100, 200, 300, 400, 500,
                    100, 200, 300, 400, 500],
    'Lane_ID':     [2, 2, 2, 3, 3, 3,   # 車両1: Lane2→Lane3に変更
                    2, 2, 2, 2, 2,
                    3, 3, 3, 3, 3],
    'Preceding':   [0, 0, 3, 3, 3, 3,   # 車両1: 最初前方なし→車両3を追従
                    1, 1, 0, 1, 1,       # 車両2: 車両1追従→前方なし→再び追従
                    0, 0, 0, 0, 0],      # 車両3: 常に前方なし
    'v_Class':     [2, 2, 2, 2, 2, 2,   # 車両1: Car
                    3, 3, 3, 3, 3,       # 車両2: Truck
                    2, 2, 2, 2, 2],      # 車両3: Car
    'time_period': [1]*16
}

df_test = pd.DataFrame(test_data)

print("テストデータ:")
print(df_test)
print("\n期待される結果:")
print("車両1: Preceding=0→NaN, Preceding=3→pair_id付与, Lane変化あり")
print("車両2: Preceding=1→pair_id付与, Preceding=0→NaN, 再びPreceding=1→新pair_id")
print("車両3: 常にPreceding=0→NaN")

# ペアID付与
print("\n--- give_pair_id テスト ---")
df_test_pairs = give_pair_id(df_test)
print(df_test_pairs[['Vehicle_ID', 'Global_Time', 'Lane_ID', 'Preceding', 'pair_id']])

# モードID付与
print("\n--- give_mode_id テスト ---")
df_test_mode = give_mode_id(df_test_pairs, df_test)
print(df_test_mode[['Vehicle_ID', 'Global_Time', 'Preceding', 'pair_id', 'mode_id']])


テストデータ:
    Vehicle_ID  Global_Time  Lane_ID  Preceding  v_Class  time_period
0            1          100        2          0        2            1
1            1          200        2          0        2            1
2            1          300        2          3        2            1
3            1          400        3          3        2            1
4            1          500        3          3        2            1
5            1          600        3          3        2            1
6            2          100        2          1        3            1
7            2          200        2          1        3            1
8            2          300        2          0        3            1
9            2          400        2          1        3            1
10           2          500        2          1        3            1
11           3          100        3          0        2            1
12           3          200        3          0        2            1
13          

In [11]:
# 有効なペアのみフィルタ
print("\n--- filter_valid_pairs テスト ---")
df_test_filtered = filter_valid_pairs(df_test_mode)
print(df_test_filtered[['Vehicle_ID', 'Global_Time', 'Preceding', 'pair_id', 'mode_id']])


--- filter_valid_pairs テスト ---
フィルタ前の行数: 16
フィルタ後の行数: 8
除外された行数: 8

ペア数: 4
    Vehicle_ID  Global_Time  Preceding  pair_id  mode_id
2            1          300          3      3.0      1.0
3            1          400          3      4.0      1.0
4            1          500          3      4.0      1.0
5            1          600          3      4.0      1.0
6            2          100          1      5.0      3.0
7            2          200          1      5.0      3.0
9            2          400          1      7.0      3.0
10           2          500          1      7.0      3.0


#### 車線・車種・追従時間